# PCam metastasis detection - full run on Colab
Set the runtime to a GPU first: **Runtime > Change runtime type > T4 GPU**.
Run the cells top to bottom. The **test set is only touched in the very last section**, once.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
!nvidia-smi -L

## 1. Get the code and install the few extra packages

In [ ]:
REPO_URL = "https://github.com/yuvxnv/pcam-metastasis-detection.git"  # <- fill in
!git clone $REPO_URL /content/repo
%cd /content/repo
!pip install -q h5py scikit-learn pandas matplotlib tqdm gdown

## 2. Download PCam (about 7.5 GB, goes to Colab's temporary disk, not the repo)

In [ ]:
!python scripts/download_data.py --root /content/data
!rm -f /content/data/pcam/*.gz

## 3. Class balance and sample grid

In [ ]:
!python scripts/explore_data.py --root /content/data

## 4. Train both models on the full training set
Best epoch is picked by **validation AUC only**. Adjust `EPOCHS` if validation AUC is still rising at the end.

In [ ]:
EPOCHS = 12
!python -m src.train --model cnn --root /content/data --epochs $EPOCHS --workers 2
!python -m src.train --model resnet18 --root /content/data --epochs $EPOCHS --workers 2

## 5. Look at the validation results (safe to repeat as often as you like)

In [ ]:
!python -m src.evaluate --model cnn --root /content/data --split val
!python -m src.evaluate --model resnet18 --root /content/data --split val
import pandas as pd
pd.read_csv("results/history_resnet18.csv")

## 6. FINAL: score the test set - ONCE per model
Only run this after you are happy with the validation work above and have decided you will not change the models.
A lock file blocks a second scoring.

In [ ]:
!python -m src.evaluate --model cnn --root /content/data --split test --final-test
!python -m src.evaluate --model resnet18 --root /content/data --split test --final-test

## 7. Download everything (results and weights are NOT committed to git)

In [ ]:
!zip -rq outputs.zip results checkpoints
from google.colab import files
files.download("outputs.zip")